# A01 - 二维稳态不可压 Navier-Stokes：顶盖驱动方腔

方程使用无量纲形式：

- 连续方程：`u_x + v_y = 0`
- x 动量：`u*u_x + v*u_y + p_x - (1/Re)*(u_xx + u_yy) = 0`
- y 动量：`u*v_x + v*v_y + p_y - (1/Re)*(v_xx + v_yy) = 0`

边界条件：顶盖 `u=1, v=0`，其余壁面 `u=0, v=0`。压力只差一个常数，所以加一个参考点 `p(0,0)=0`。

说明：这是从零写 PINN 版本的教学算例。要得到高精度涡结构，需要更长训练、更多采样和更好的优化策略。

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(101)
device = "cuda" if torch.cuda.is_available() else "cpu"
Re = 100.0
device

In [ ]:
class MLP(nn.Module):
    def __init__(self, width=80, depth=5):
        super().__init__()
        layers = [nn.Linear(2, width), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.Tanh()]
        layers += [nn.Linear(width, 3)]
        self.net = nn.Sequential(*layers)

    def forward(self, xy):
        return self.net(xy)

def grad(y, x):
    return torch.autograd.grad(y, x, torch.ones_like(y), create_graph=True)[0]

model = MLP().to(device)

In [ ]:
def ns_residual(x, y):
    x.requires_grad_(True)
    y.requires_grad_(True)
    out = model(torch.cat([x, y], dim=1))
    u, v, p = out[:, 0:1], out[:, 1:2], out[:, 2:3]
    u_x, u_y = grad(u, x), grad(u, y)
    v_x, v_y = grad(v, x), grad(v, y)
    p_x, p_y = grad(p, x), grad(p, y)
    u_xx, u_yy = grad(u_x, x), grad(u_y, y)
    v_xx, v_yy = grad(v_x, x), grad(v_y, y)
    continuity = u_x + v_y
    mom_x = u * u_x + v * u_y + p_x - (u_xx + u_yy) / Re
    mom_y = u * v_x + v * v_y + p_y - (v_xx + v_yy) / Re
    return continuity, mom_x, mom_y

def boundary_points(n=128):
    s = torch.rand(n, 1, device=device)
    x_top, y_top = s, torch.ones_like(s)
    x_bottom, y_bottom = s, torch.zeros_like(s)
    x_left, y_left = torch.zeros_like(s), s
    x_right, y_right = torch.ones_like(s), s
    return (x_top, y_top), (x_bottom, y_bottom), (x_left, y_left), (x_right, y_right)

def bc_loss():
    (xt, yt), (xb, yb), (xl, yl), (xr, yr) = boundary_points()
    top = model(torch.cat([xt, yt], dim=1))
    bottom = model(torch.cat([xb, yb], dim=1))
    left = model(torch.cat([xl, yl], dim=1))
    right = model(torch.cat([xr, yr], dim=1))
    loss_top = torch.mean((top[:, 0:1] - 1)**2 + top[:, 1:2]**2)
    loss_wall = torch.mean(bottom[:, :2]**2) + torch.mean(left[:, :2]**2) + torch.mean(right[:, :2]**2)
    p_ref = model(torch.tensor([[0.0, 0.0]], device=device))[:, 2:3]
    return loss_top + loss_wall + torch.mean(p_ref**2)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
history = []

for epoch in range(12000):
    x = torch.rand(1024, 1, device=device)
    y = torch.rand(1024, 1, device=device)
    cont, mx, my = ns_residual(x, y)
    loss_pde = torch.mean(cont**2) + torch.mean(mx**2) + torch.mean(my**2)
    loss_bc = bc_loss()
    loss = loss_pde + 20.0 * loss_bc
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    history.append([loss.item(), loss_pde.item(), loss_bc.item()])
    if epoch % 2000 == 0:
        print(f"epoch={epoch:5d}, total={loss.item():.3e}, pde={loss_pde.item():.3e}, bc={loss_bc.item():.3e}")

In [ ]:
nx = ny = 80
xs = torch.linspace(0, 1, nx, device=device)
ys = torch.linspace(0, 1, ny, device=device)
X, Y = torch.meshgrid(xs, ys, indexing="ij")
xy = torch.cat([X.reshape(-1, 1), Y.reshape(-1, 1)], dim=1)
with torch.no_grad():
    out = model(xy)
    U = out[:, 0].reshape(nx, ny)
    V = out[:, 1].reshape(nx, ny)
    P = out[:, 2].reshape(nx, ny)
    speed = torch.sqrt(U**2 + V**2)

plt.figure(figsize=(13, 4))
for i, (field, title) in enumerate([(speed, "speed"), (P, "pressure"), (U, "u velocity")], 1):
    plt.subplot(1, 3, i)
    plt.imshow(field.cpu().T, origin="lower", extent=[0,1,0,1])
    plt.colorbar()
    plt.title(title)
plt.tight_layout()
plt.show()

plt.figure(figsize=(5, 4))
plt.semilogy(torch.tensor(history)[:, 0], label="total")
plt.semilogy(torch.tensor(history)[:, 1], label="pde")
plt.semilogy(torch.tensor(history)[:, 2], label="bc")
plt.legend()
plt.title("loss")
plt.show()